In [ ]:
import os
import glob
import pickle
import pandas as pd
import numpy as np
import networkx as nx

from dask.diagnostics import ProgressBar
import scipy.sparse
from scipy.stats import wilcoxon
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import pearsonr
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from dask.distributed import Client, LocalCluster

from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell

import seaborn as sns
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from pyscenic.cli.utils import load_exp_matrix
from pyscenic.prune import prune2df, df2regulons
from pyscenic.export import export2loom
from pyscenic.aucell import aucell
#from pyscenic.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies
from dask.diagnostics import ProgressBar

In [ ]:
## 导入数据库

In [ ]:
DATABASE_FOLDER = '/data/input/Files/xiamuqing/scenic_dps/human/'
OUTPUT_DIR = "/data/work/06_pyscenic/scenic_exn/"

adata_path = '/data/work/sc_reference/EXN_harmony_nasa_res0_5_annotated.h5ad'
DATABASES_GLOB = os.path.join(DATABASE_FOLDER, "*rankings.feather")
# Supply the human-symbol cisTarget rankings used for this analysis.

MOTIF_ANNOTATIONS_FNAME = os.path.join(DATABASE_FOLDER, "motifs-v9-nr.hgnc-m0.001-o0.0.tbl.txt")
# https://resources.aertslab.org/cistarget/motif2tf/

REGULONS_FNAME = os.path.join(OUTPUT_DIR, "regulons.p")
MOTIFS_FNAME = os.path.join(OUTPUT_DIR, "motifs.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)
N_WORKERS = 4  # adjust to available memory and CPU resources


In [ ]:
# 加载数据
adata = sc.read_h5ad(adata_path)
#sc.pp.subsample(adata, fraction=0.1)
if 'counts' not in adata.layers:
    raise ValueError("Provide raw counts in adata.layers['counts']; normalized X is not a count input.")

# 准备表达矩阵
ex_matrix = pd.DataFrame(
    adata.layers['counts'].toarray() if scipy.sparse.issparse(adata.layers['counts']) else adata.layers['counts'],
    columns=adata.var_names,
    index=adata.obs_names
)

# 加载TF列表
tf_list_path = "/data/work/06_pyscenic/hs_hgnc_tfs.txt"
tf_names = load_tf_names(tf_list_path)

In [ ]:
client = Client(n_workers=N_WORKERS, threads_per_worker=1)
# 运行 GRNBoost2
adjacencies = grnboost2(
    expression_data=ex_matrix,
    tf_names=tf_names,
    client_or_address=client,
    verbose=True# 显式传递 client
)

In [ ]:
# 保存结果
adjacencies.head()
adjacencies.to_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_adjacencies.csv"), index=False)
print(f"生成的调控关系数量: {len(adjacencies)}")

In [ ]:
db_fnames = glob.glob(DATABASES_GLOB)
def name(fname):
    return os.path.splitext(os.path.basename(fname))[0]
dbs = [RankingDatabase(fname=fname, name=name(fname)) for fname in db_fnames]
dbs

In [ ]:
modules = list(modules_from_adjacencies(adjacencies, ex_matrix))


In [ ]:
# 计算富集基序
# Calculate a list of enriched motifs and the corresponding target genes for all modules.
with ProgressBar():
    df = prune2df(dbs, modules, MOTIF_ANNOTATIONS_FNAME)

# Create regulons from this table of enriched motifs.
regulons = df2regulons(df)

In [ ]:
# Save the enriched motifs and the discovered regulons to disk.
df.to_csv(MOTIFS_FNAME)
with open(REGULONS_FNAME, "wb") as f:
    pickle.dump(regulons, f)

num_regulons = len(regulons)
print(f"Number of regulons: {num_regulons}")

In [ ]:
## 可视化

In [ ]:
matrix = ex_matrix.copy()


In [ ]:
matrix = matrix.loc[adata.obs.index, :]
matrix = matrix.loc[:, adata.var.index]

# Score inferred regulons with AUCell and retain the cell-by-regulon matrix.
auc_mtx = aucell(matrix, regulons)
auc_mtx.to_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_auc_matrix.csv"))

In [ ]:
auc_mtx = pd.read_csv(os.path.join(OUTPUT_DIR, "EXN_nasa_auc_matrix.csv"),index_col=0)

In [ ]:
annotations = adata.obs["annotation"]

# 对齐注释与当前auc_mtx的细胞
filtered_annotations = annotations[annotations.index.isin(auc_mtx.index)]

# 去除注释中的“Other”标签细胞
filtered_annotations = filtered_annotations[filtered_annotations != "Other"]

# 按细胞类型分组计算平均AUC（保留分组顺序）
grouped_auc_mtx = (
    auc_mtx
    .assign(cell_type=filtered_annotations)  # 添加细胞类型列
    .groupby('cell_type', observed=True)
    .mean()
    .reindex(pd.unique(annotations.dropna()))  # 按原始类别顺序排序
    .dropna()  # 移除空分组
)

In [ ]:
## 富集因子分组热图

In [ ]:
g = sns.clustermap(grouped_auc_mtx.T, col_cluster=True, figsize=(12, 10),
                   col_colors=None,  # 不需要行颜色条
                   row_cluster=False)  # 不对行进行聚类

g.ax_heatmap.set_xticklabels(grouped_auc_mtx.index, rotation=90)
g.savefig(os.path.join(OUTPUT_DIR, "EXN_nasa_motif_heatmap_grouped_90.pdf"), format="pdf", bbox_inches="tight")